# TF-IDF Text Baseline From Precomputed CSV (IEMOCAP)

Uses `extracted_features/text/tfidf_features.csv` generated by the extractor notebook.

## 1) Setup and Data Loading
This section imports packages, loads the TF-IDF feature CSV, isolates feature columns, and creates `X_train`, `X_test`, `y_train`, and `y_test`.


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

repo_root = Path.cwd().resolve().parents[1] if Path.cwd().name == 'TreeBased' else Path.cwd().resolve()
csv_path = repo_root / 'extracted_features' / 'text' / 'tfidf_features.csv'

if not csv_path.exists():
    raise FileNotFoundError(f'Missing CSV: {csv_path}')

df = pd.read_csv(csv_path)

metadata_cols = ['path', 'session', 'method', 'gender', 'emotion', 'n_annotators', 'agreement', 'utt_id', 'text', 'split']
feature_cols = [c for c in df.columns if c.startswith('tfidf_')]

if not feature_cols:
    raise ValueError('No TF-IDF feature columns found (expected columns starting with tfidf_).')

work_df = df[['emotion'] + feature_cols].dropna(axis=0, how='any').copy()
X = work_df[feature_cols]
y = work_df['emotion'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Dataset: {csv_path.name}')
print(f'Rows used: {len(work_df)}')
print(f'Feature count: {len(feature_cols)}')
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print('Class distribution (train):')
print(y_train.value_counts())


Dataset: tfidf_features.csv
Rows used: 7532
Feature count: 9155
Train shape: (6025, 9155)
Test shape: (1507, 9155)
Class distribution (train):
emotion
fru    1479
neu    1366
ang     882
sad     867
exc     833
hap     476
sur      86
fea      32
dis       2
oth       2
Name: count, dtype: int64


## 2) Baseline Logistic Regression (Holdout Test)
Train a baseline classifier on the full TF-IDF feature set and evaluate on the test split.


In [6]:
clf = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver='liblinear',
    random_state=42,
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy: {acc:.4f}')
print(f'Macro F1:  {f1:.4f}')
print('\nClassification report:\n')
print(classification_report(y_test, y_pred))


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Accuracy: 0.5202
Macro F1:  0.3816

Classification report:

              precision    recall  f1-score   support

         ang       0.57      0.64      0.60       221
         dis       0.00      0.00      0.00         0
         exc       0.51      0.52      0.52       208
         fea       0.36      0.62      0.45         8
         fru       0.58      0.55      0.57       370
         hap       0.38      0.37      0.38       119
         neu       0.50      0.42      0.46       342
         oth       0.00      0.00      0.00         1
         sad       0.56      0.59      0.58       217
         sur       0.19      0.48      0.27        21

    accuracy                           0.52      1507
   macro avg       0.37      0.42      0.38      1507
weighted avg       0.53      0.52      0.52      1507



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\

## 3) Cross-Validation on Training Split
Run 5-fold stratified CV to get a more stable estimate than one holdout split.


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(clf, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
cv_f1 = cross_val_score(clf, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)

print(f'CV Accuracy (mean+-std): {np.mean(cv_acc):.4f} +- {np.std(cv_acc):.4f}')
print(f'CV Macro F1 (mean+-std):  {np.mean(cv_f1):.4f} +- {np.std(cv_f1):.4f}')


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy (mean+-std): 0.5127 +- 0.0127
CV Macro F1 (mean+-std):  0.3881 +- 0.0154


## 4) Filtered Emotion Experiment
Rerun the pipeline after removing `fear`, `disgust`, and `other` to see how performance changes on a reduced label set.


In [10]:
# Remove selected emotion classes and rerun the same workflow
remove_emotions = {'dis', 'fea', 'oth', 'sur'}

mask_keep = ~y.isin(remove_emotions)
X_filtered = X.loc[mask_keep]
y_filtered = y.loc[mask_keep]

print('Removed emotions:', sorted(remove_emotions))
print(f'Rows kept: {len(y_filtered)} / {len(y)}')
print('Filtered class distribution:')
print(y_filtered.value_counts())

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_filtered,
    y_filtered,
    test_size=0.2,
    random_state=42,
    stratify=y_filtered,
)

clf_filtered = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver='liblinear',
    random_state=42,
)

clf_filtered.fit(X_train_f, y_train_f)
y_pred_f = clf_filtered.predict(X_test_f)

acc_f = accuracy_score(y_test_f, y_pred_f)
f1_f = f1_score(y_test_f, y_pred_f, average='macro')

print('Filtered holdout results:')
print(f'Accuracy: {acc_f:.4f}')
print(f'Macro F1:  {f1_f:.4f}')
print('Classification report (filtered):')
print(classification_report(y_test_f, y_pred_f))

skf_f = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc_f = cross_val_score(clf_filtered, X_train_f, y_train_f, cv=skf_f, scoring='accuracy', n_jobs=-1)
cv_f1_f = cross_val_score(clf_filtered, X_train_f, y_train_f, cv=skf_f, scoring='f1_macro', n_jobs=-1)

print('Filtered CV results (train split only):')
print(f'CV Accuracy (mean+-std): {np.mean(cv_acc_f):.4f} +- {np.std(cv_acc_f):.4f}')
print(f'CV Macro F1 (mean+-std):  {np.mean(cv_f1_f):.4f} +- {np.std(cv_f1_f):.4f}')


Removed emotions: ['dis', 'fea', 'oth']
Rows kept: 7487 / 7532
Filtered class distribution:
emotion
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
sur     107
Name: count, dtype: int64


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Filtered holdout results:
Accuracy: 0.5234
Macro F1:  0.4998
Classification report (filtered):
              precision    recall  f1-score   support

         ang       0.56      0.65      0.60       221
         exc       0.53      0.55      0.54       208
         fru       0.57      0.53      0.55       370
         hap       0.36      0.33      0.35       119
         neu       0.50      0.40      0.44       342
         sad       0.56      0.64      0.60       217
         sur       0.31      0.67      0.42        21

    accuracy                           0.52      1498
   macro avg       0.48      0.54      0.50      1498
weighted avg       0.52      0.52      0.52      1498

Filtered CV results (train split only):
CV Accuracy (mean+-std): 0.5168 +- 0.0263
CV Macro F1 (mean+-std):  0.4814 +- 0.0277
